In [34]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# -----------------------------
# Paths (adjust if needed)
# -----------------------------
BASE_DIR = Path(".").resolve()   # notebook is inside PROF
INPUT_DIR = BASE_DIR.parent / "data" / "stx_xlsx"
MAPPINGS_DIR = BASE_DIR / "mappings" / "mappings_hex"
OUT_DIR = BASE_DIR / "prof_components_extracted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Only keep years <= 2024
MAX_YEAR = 2024

# Variable logic
PRIORITY_VARS = {"REVT", "XRD", "BE", "MIB"}
SUM_VARS = {"COGS", "XSGA_COMPONENTS", "XINT"}

ALL_VARS = ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT", "BE", "MIB"]



In [ ]:

# -----------------------------
# Helpers
# -----------------------------
def parse_year(colname: str) -> Optional[int]:
    """
    Extract year from column name.
    Handles headers like '2024', 'FY2024', '31-12-2024', '2024-12-31', etc.
    Returns int year or None.
    """
    if colname is None:
        return None
    s = str(colname).strip()
    # Find a 4-digit year 19xx/20xx
    m = re.search(r"(19\d{2}|20\d{2})", s)
    if not m:
        return None
    return int(m.group(1))


def to_float(x):
    """Robust numeric parsing incl. European formats and thousand separators."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)

    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    # remove spaces / NBSP
    s = s.replace("\u00a0", "").replace(" ", "")

    # if both comma and dot exist, assume comma is thousands sep
    if "," in s and "." in s:
        s = s.replace(",", "")
    else:
        # if only comma exists, treat comma as decimal separator
        if "," in s and "." not in s:
            s = s.replace(",", ".")

    try:
        return float(s)
    except ValueError:
        return np.nan



def detect_header_row(xlsx_path: Path, sheet_name, max_scan_rows: int = 80) -> Optional[int]:
    """
    Returns 0-indexed header row where column A equals 'Field Name'.
    Reads only column A to avoid out-of-bounds issues.
    """
    scan = pd.read_excel(
        xlsx_path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_scan_rows,
        usecols="A",    
        engine="openpyxl",
    )

    for i in range(len(scan)):
        a0 = scan.iloc[i, 0]
        a0 = "" if pd.isna(a0) else str(a0).strip()
        if a0.lower() == "field name":
            return i

    return None


def read_sheet_all_years(xlsx_path: Path, sheet_name: str) -> pd.DataFrame:
    """
    Reads the sheet using the detected 'Field Name' row as the header.
    Returns a DataFrame with column 0 as labels and remaining columns as years/dates.
    """
    header_row = detect_header_row(xlsx_path, sheet_name)
    n = 0
    if header_row is None:
        # fallback: try your old behavior
        print("NO HEADER ROW DETECTED, FALLBACK TO SKIPROWS=18")
        n += 1
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=18)
    else:
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=header_row)

    if df.empty or df.shape[1] == 0:
        return df

    label_col = df.columns[0]
    df[label_col] = df[label_col].astype(str).str.strip()
    df = df[df[label_col] != ""].copy()
    return df, n


def keep_year_cols_upto(df: pd.DataFrame, max_year: int) -> Tuple[pd.DataFrame, List[int]]:
    """
    Keeps only the year columns with parsed year <= max_year.
    Returns (df_filtered, years_sorted_desc_or_asc).
    """
    if df.empty:
        return df, []

    label_col = df.columns[0]
    year_cols = []
    years = []
    for c in df.columns[1:]:
        y = parse_year(c)
        if y is not None and y <= max_year:
            year_cols.append(c)
            years.append(y)

    # If multiple columns map to same year (rare), keep them all for now; we’ll aggregate later
    keep_cols = [label_col] + year_cols
    df2 = df[keep_cols].copy()

    # Standardize columns to year ints (may create duplicates)
    new_cols = [label_col] + [parse_year(c) for c in year_cols]
    df2.columns = new_cols

    # If duplicate year columns exist, collapse by taking first non-null across them (per row)
    # e.g., two columns both interpreted as 2024
    if len(set(new_cols[1:])) != len(new_cols[1:]):
        # group by year and take first non-null across duplicate columns
        label = df2.columns[0]
        out = df2[[label]].copy()
        for y in sorted(set(new_cols[1:])):
            cols_y = [c for c in df2.columns[1:] if c == y]
            block = df2[cols_y].applymap(to_float)
            out[y] = block.bfill(axis=1).iloc[:, 0]
        df2 = out

    years_sorted = sorted([c for c in df2.columns[1:] if isinstance(c, int)])
    return df2, years_sorted


def extract_label_series_with_duplicates(df_years: pd.DataFrame, row_label: str) -> pd.Series:
    """
    Extract a time series for a given row_label, handling duplicates:
    if multiple rows have same label, for each year take first non-null from top-most row,
    else next row, etc.
    """
    if df_years.empty:
        return pd.Series(dtype=float)

    label_col = df_years.columns[0]
    hits = df_years[df_years[label_col] == row_label]
    if hits.empty:
        return pd.Series(dtype=float)

    # numeric block: rows x years
    block = hits.drop(columns=[label_col]).applymap(to_float)

    # pick first non-null down the rows for each year (top-most wins)
    # bfill down rows then take first row
    picked = block.bfill(axis=0).iloc[0]
    picked.name = row_label
    return picked


def priority_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """
    Given a list of label-series in priority order, pick first non-null per year.
    """
    if not series_list:
        return pd.Series(dtype=float)

    df = pd.concat(series_list, axis=1)  # years index
    out = df.bfill(axis=1).iloc[:, 0]
    return out


def sum_across_labels(series_list: List[pd.Series]) -> pd.Series:
    """
    Sum series across labels (align on years).
    """
    if not series_list:
        return pd.Series(dtype=float)
    df = pd.concat(series_list, axis=1)
    return df.sum(axis=1, min_count=1)


def load_mapping(mapping_path: Path) -> Dict:
    return json.loads(mapping_path.read_text(encoding="utf-8"))

def get_company_metadata(xlsx_path: Path) -> Tuple[str, str]:
    """
    Reads:
      - Company name from cell B2
      - TRBC Industry Group from cell B5
    from the first sheet in the workbook.

    Returns (company_name_without_ticker, industry).
    """
    # Read only first 6 rows and columns A-B from sheet 0
    meta = pd.read_excel(xlsx_path, sheet_name=0, header=None, usecols="A:B", nrows=6)

    raw_name = "" if pd.isna(meta.iat[1, 1]) else str(meta.iat[1, 1]).strip()  # B2
    industry  = "" if pd.isna(meta.iat[4, 1]) else str(meta.iat[4, 1]).strip()  # B5

    # Remove trailing ticker in parentheses, e.g. "Aker BP ASA (AKRBP.OL)" -> "Aker BP ASA"
    company_name = re.sub(r"\s*\([^)]*\)\s*$", "", raw_name).strip()

    return company_name, industry

def trbc_industry_to_sector(industry: str) -> str:
    # Explicit mapping based on your observed 33 industry groups
    mapping = {
    # Energy
    "Oil & Gas": "Energy",
    "Oil & Gas Related Equipment and Services": "Energy",
    "Renewable Energy": "Energy",

    # Industrials
    "Freight & Logistics Services": "Industrials",
    "Machinery, Tools, Heavy Vehicles, Trains & Ships": "Industrials",
    "Construction & Engineering": "Industrials",
    "Professional & Commercial Services": "Industrials",
    "Aerospace & Defense": "Industrials",
    "Passenger Transportation Services": "Industrials",
    "Diversified Industrial Goods Wholesale": "Industrials",
    "Construction Materials": "Industrials",
    "Transport Infrastructure": "Industrials",

    # Consumer Staples
    "Food & Tobacco": "Consumer Staples",
    "Beverages": "Consumer Staples",
    "Food & Drug Retailing": "Consumer Staples",
    "Household Goods": "Consumer Staples",
    "Personal & Household Products & Services": "Consumer Staples",

    # Consumer Discretionary
    "Specialty Retailers": "Consumer Discretionary",
    "Diversified Retail": "Consumer Discretionary",
    "Automobiles & Auto Parts": "Consumer Discretionary",
    "Hotels & Entertainment Services": "Consumer Discretionary",
    "Homebuilding & Construction Supplies": "Consumer Discretionary",
    "Textiles & Apparel": "Consumer Discretionary",
    "Leisure Products": "Consumer Discretionary",
    "Consumer Goods Conglomerates": "Consumer Discretionary",

    # Health Care
    "Biotechnology & Medical Research": "Health Care",
    "Pharmaceuticals": "Health Care",
    "Healthcare Equipment & Supplies": "Health Care",
    "Healthcare Providers & Services": "Health Care",

    # Financials
    "Banking Services": "Financials",
    "Investment Banking & Investment Services": "Financials",
    "Investment Holding Companies": "Financials",
    "Insurance": "Financials",

    # Materials
    "Metals & Mining": "Materials",
    "Chemicals": "Materials",
    "Paper & Forest Products": "Materials",
    "Containers & Packaging": "Materials",

    # Information Technology
    "Software & IT Services": "Information Technology",
    "Semiconductors & Semiconductor Equipment": "Information Technology",
    "Electronic Equipment & Parts": "Information Technology",
    "Computers, Phones & Household Electronics": "Information Technology",
    "Integrated Hardware & Software": "Information Technology",
    "Office Equipment": "Information Technology",
    "Communications & Networking": "Communication Services",

    # Communication Services
    "Media & Publishing": "Communication Services",
    "Telecommunications Services": "Communication Services",

    # Utilities
    "Electric Utilities & IPPs": "Utilities",

    # Real Estate
    "Real Estate Operations": "Real Estate",

    # Education (typically folded into Industrials or Consumer Discretionary depending on use case)
    "Professional & Business Education": "Industrials",
    "Miscellaneous Educational Service Providers": "Industrials",
    }

    if industry is None:
        return "Other"
    key = str(industry).strip()
    return mapping.get(key, "Other")


In [36]:


# -----------------------------
# Main extraction per firm
# -----------------------------
def extract_firm_components(xlsx_path: Path, mapping_path: Path, max_year: int = 2024) -> pd.DataFrame:
    mapping = load_mapping(mapping_path)
    variables = mapping.get("variables", [])

    # Cache sheet dfs (year-filtered) because we’ll read many labels per sheet
    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()

    extracted: Dict[str, pd.Series] = {}

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in ALL_VARS:
            continue

        choices = var_item.get("final_choice", [])  # list of {sheet_name,row_label}

        # Empty final_choice -> return empty series for now (later fill with 0)
        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        # Extract a series for each chosen label (with duplicate-row handling)
        label_series_list: List[pd.Series] = []

        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw, n = read_sheet_all_years(xlsx_path, sheet)
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        # Apply variable logic
        if var in PRIORITY_VARS:
            s_var = priority_across_labels(label_series_list)
        elif var in SUM_VARS:
            s_var = sum_across_labels(label_series_list)
        else:
            # fallback (shouldn't happen)
            s_var = sum_across_labels(label_series_list)

        extracted[var] = s_var
        years_union.update(list(s_var.index))

    # Build final DF over all years seen
    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ALL_VARS:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    # Fill missing series (e.g., final_choice=[]) with 0 as per your rule for later calculations
    # If you prefer leaving as NaN for now, change fillna(0.0) to keep NaN.
    df_out = df_out.fillna(0.0)

    return df_out, n



In [37]:
# -----------------------------
# Run extraction for all firms
# -----------------------------

EXCHANGES = ["cox", "hex", "obx", "stx", "isx"]

BASE_DATA_DIR = BASE_DIR.parent / "data"
BASE_MAP_DIR  = BASE_DIR / "mappings"

total_ok, total_skip = 0, 0
header_not_found_total = 0

for ex in EXCHANGES:

    INPUT_DIR = BASE_DATA_DIR / f"{ex}_xlsx"
    MAPPINGS_DIR = BASE_MAP_DIR / f"mappings_{ex}"

    if not INPUT_DIR.exists():
        print(f"[skip exchange] missing INPUT_DIR: {INPUT_DIR}")
        continue
    if not MAPPINGS_DIR.exists():
        print(f"[skip exchange] missing MAPPINGS_DIR: {MAPPINGS_DIR}")
        continue

    mapping_files = sorted(MAPPINGS_DIR.glob("*.json"))
    print(f"\n=== {ex.upper()} ===")
    print(f"Found {len(mapping_files)} mapping files in {MAPPINGS_DIR}")

    n_ok, n_skip = 0, 0
    last_n = 0

    for mp in mapping_files:
        firm_id = mp.stem
        xlsx_path = INPUT_DIR / f"{firm_id}.xlsx"
        if not xlsx_path.exists():
            n_skip += 1
            continue

        df, last_n = extract_firm_components(xlsx_path, mp, max_year=MAX_YEAR)

        # Make Year a normal column instead of index
        df = df.reset_index()

        # firm metadata
        company_name, industry = get_company_metadata(xlsx_path)
        sector = trbc_industry_to_sector(industry)

        df.insert(1, "Ticker", firm_id)
        df.insert(2, "CompanyName", company_name)
        df.insert(3, "Industry", industry)
        df.insert(4, "Sector", sector)

        out_path = OUT_DIR / f"{firm_id}.csv"
        df.to_csv(out_path, index=False)
        n_ok += 1

    print(f"Done. Extracted {n_ok} firms. Skipped {n_skip} missing workbooks.")
    print(f"Header not found in {last_n} sheets.")
    total_ok += n_ok
    total_skip += n_skip
    header_not_found_total += last_n


print(f"\nALL DONE. Extracted {total_ok} firms total. Skipped {total_skip} missing workbooks.")
print(f"Header not found (sum across exchanges): {header_not_found_total} sheets.")


=== COX ===
Found 87 mapping files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/mappings/mappings_cox


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin

Done. Extracted 86 firms. Skipped 1 missing workbooks.
Header not found in 0 sheets.

=== HEX ===
Found 119 mapping files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/mappings/mappings_hex


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin

Done. Extracted 118 firms. Skipped 1 missing workbooks.
Header not found in 0 sheets.

=== OBX ===
Found 148 mapping files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/mappings/mappings_obx


...

Done. Extracted 148 firms. Skipped 0 missing workbooks.
Header not found in 0 sheets.

=== STX ===
Found 262 mapping files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/mappings/mappings_stx


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin

Done. Extracted 260 firms. Skipped 2 missing workbooks.
Header not found in 0 sheets.

=== ISX ===
Found 22 mapping files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/mappings/mappings_isx


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin

Done. Extracted 20 firms. Skipped 2 missing workbooks.
Header not found in 0 sheets.

ALL DONE. Extracted 632 firms total. Skipped 6 missing workbooks.
Header not found (sum across exchanges): 0 sheets.


In [38]:
BASE_DIR = Path(".").resolve()
IN_DIR = BASE_DIR / "prof_components_extracted"

for fp in sorted(IN_DIR.glob("*.csv")):
    df = pd.read_csv(fp, index_col=0)

    # Expense-like items -> positive
    for c in ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT"]:
        if c in df.columns:
            df[c] = df[c].abs()

    numer = df["REVT"] - df["COGS"] - (df["XSGA_COMPONENTS"] - df["XRD"]) - df["XINT"]
    denom = (df["BE"] + df["MIB"]).replace({0.0: np.nan})

    df["PROF"] = numer / denom
    df.to_csv(fp)

In [53]:


# -----------------------------
# Paths (adjust exchange if needed)
# -----------------------------
BASE_DIR = Path(".").resolve()

EXCHANGE = "isx"   # change to isx / cox / stx / obx / hex

INPUT_XLSX_DIR = BASE_DIR.parent / "data" / f"{EXCHANGE}_xlsx"
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / f"da_mappings_{EXCHANGE}"
CSV_DIR = BASE_DIR / "prof_components_extracted"

MAX_YEAR = 2024


# -----------------------------
# D&A extraction per firm
# -----------------------------
def extract_firm_da_components(
    xlsx_path: Path,
    da_mapping_path: Path,
    max_year: int = 2024,
):
    da_mapping = load_mapping(da_mapping_path)
    variables = da_mapping.get("variables", [])

    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()
    extracted = {
        "COGS_DA": pd.Series(dtype=float),
        "XSGA_DA": pd.Series(dtype=float),
    }

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in {"COGS_DA", "XSGA_DA"}:
            continue

        choices = var_item.get("final_choice", [])

        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        label_series_list: List[pd.Series] = []

        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw, _ = read_sheet_all_years(xlsx_path, sheet)
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        s_var = sum_across_labels(label_series_list)
        extracted[var] = s_var
        years_union.update(list(s_var.index))

    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ["COGS_DA", "XSGA_DA"]:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    df_out = df_out.fillna(0.0)

    return df_out.reset_index()


# -----------------------------
# Add D&A to existing CSVs
# -----------------------------
csv_files = sorted(CSV_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {CSV_DIR}")

n_ok = 0
n_skip_missing_xlsx = 0
n_skip_missing_da = 0
n_error = 0

for csv_path in csv_files:
    firm_id = csv_path.stem
    xlsx_path = INPUT_XLSX_DIR / f"{firm_id}.xlsx"
    da_mapping_path = DA_MAPPINGS_DIR / f"{firm_id}.json"

    if not xlsx_path.exists():
        print(f"[skip] {firm_id}: missing xlsx")
        n_skip_missing_xlsx += 1
        continue

    if not da_mapping_path.exists():
        print(f"[skip] {firm_id}: missing D&A mapping")
        n_skip_missing_da += 1
        continue

    try:
        # existing extracted csv
        df_csv = pd.read_csv(csv_path)

        # enforce numeric year
        df_csv["Year"] = pd.to_numeric(df_csv["Year"], errors="coerce").astype("Int64")
        df_csv = df_csv[df_csv["Year"].notna()].copy()
        df_csv["Year"] = df_csv["Year"].astype(int)

        # extract D&A additions
        df_da = extract_firm_da_components(xlsx_path, da_mapping_path, max_year=MAX_YEAR)

        # merge
        df = df_csv.merge(df_da, on="Year", how="left")
        df["COGS_DA"] = df["COGS_DA"].fillna(0.0)
        df["XSGA_DA"] = df["XSGA_DA"].fillna(0.0)

        # make sure base cols are numeric
        for c in ["REVT", "COGS", "XSGA_COMPONENTS", "XRD", "XINT", "BE", "MIB", "COGS_DA", "XSGA_DA"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

        # add D&A to existing extracted components
        df["COGS"] = df["COGS"] + df["COGS_DA"]
        df["XSGA_COMPONENTS"] = df["XSGA_COMPONENTS"] + df["XSGA_DA"]

        # recompute PROF
        numer = df["REVT"] - df["COGS"] - (df["XSGA_COMPONENTS"] - df["XRD"]) - df["XINT"]
        denom = (df["BE"] + df["MIB"]).replace({0.0: np.nan})
        df["PROF"] = numer / denom

        # overwrite csv
        df.to_csv(csv_path, index=False)

        n_ok += 1

    except Exception as e:
        print(f"[error] {firm_id}: {type(e).__name__}: {e}")
        n_error += 1

print(f"Done. Updated {n_ok} files.")
print(f"Skipped missing xlsx: {n_skip_missing_xlsx}")
print(f"Skipped missing D&A mapping: {n_skip_missing_da}")
print(f"Errors: {n_error}")

Found 635 CSV files in /Users/simenoiseth/Desktop/Equity_Quality_Under_Uncertainty/PROF/prof_components_extracted
[skip] 20202.OL: missing xlsx
[skip] 8TRA.ST: missing xlsx
[skip] AAB.CO: missing xlsx
[skip] AAK.ST: missing xlsx
[skip] ABB.ST: missing xlsx
[skip] ABL.OL: missing xlsx
[skip] ACADE.ST: missing xlsx
[skip] ACAST.ST: missing xlsx
[skip] ACELP.ST: missing xlsx
[skip] ACG1V.HE: missing xlsx
[skip] ACRIa.ST: missing xlsx
[skip] ACTI.ST: missing xlsx
[skip] ADDTb.ST: missing xlsx
[skip] AFAGR.HE: missing xlsx
[skip] AFGA.OL: missing xlsx
[skip] AFK.OL: missing xlsx
[skip] AFRY.ST: missing xlsx
[skip] AGATE.CO: missing xlsx
[skip] AGFEb.CO: missing xlsx
[skip] AKAST.OL: missing xlsx
[skip] AKBM.OL: missing xlsx
[skip] AKH.OL: missing xlsx
[skip] AKRBP.OL: missing xlsx
[skip] AKSOA.OL: missing xlsx
[skip] AKVA.OL: missing xlsx
[skip] ALFA.ST: missing xlsx
[skip] ALIFb.ST: missing xlsx
[skip] ALIG.ST: missing xlsx
[skip] ALIVsdb.ST: missing xlsx
[skip] ALKb.CO: missing xlsx
[skip

/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)


[skip] EIOF.OL: missing xlsx
[skip] EKTAb.ST: missing xlsx
[skip] ELABS.OL: missing xlsx
[skip] ELANb.ST: missing xlsx
[skip] ELEAV.HE: missing xlsx
[skip] ELISA.HE: missing xlsx
[skip] ELK.OL: missing xlsx
[skip] ELMRA.OL: missing xlsx
[skip] ELO.OL: missing xlsx
[skip] ELON.ST: missing xlsx
[skip] ELTEL.ST: missing xlsx
[skip] ELUXb.ST: missing xlsx
[skip] EMBLA.CO: missing xlsx
[skip] EMGS.OL: missing xlsx
[skip] ENDUR.OL: missing xlsx
[skip] ENEA.ST: missing xlsx
[skip] ENENTO.HE: missing xlsx
[skip] ENGCONb.ST: missing xlsx
[skip] ENH.OL: missing xlsx
[skip] ENRO.ST: missing xlsx
[skip] ENSG.CO: missing xlsx
[skip] ENSU.OL: missing xlsx
[skip] ENTRA.OL: missing xlsx
[skip] ENVIP.OL: missing xlsx
[skip] EOLUb.ST: missing xlsx
[skip] EPEND.ST: missing xlsx
[skip] EPIRa.ST: missing xlsx
[skip] EPISb.ST: missing xlsx
[skip] EPR.OL: missing xlsx
[skip] EPROb.ST: missing xlsx
[skip] EQNR.OL: missing xlsx
[skip] EQVA.OL: missing xlsx
[skip] ERICb.ST: missing xlsx
[skip] ESENSE.HE: missin

/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)


[skip] HARBb.CO: missing xlsx
[skip] HARVIA.HE: missing xlsx
[skip] HAUTO.OL: missing xlsx
[skip] HAVI.OL: missing xlsx
[skip] HBC.OL: missing xlsx
[skip] HEX.OL: missing xlsx
[skip] HHDC.CO: missing xlsx
[skip] HIAB.HE: missing xlsx
[skip] HKFOODS.HE: missing xlsx
[skip] HLUNa.CO: missing xlsx
[skip] HMb.ST: missing xlsx
[skip] HONBS.HE: missing xlsx
[skip] HPUR.OL: missing xlsx
[skip] HSHP.OL: missing xlsx
[skip] HUH1V.HE: missing xlsx
[skip] HUMAN.ST: missing xlsx
[skip] HUSCO.CO: missing xlsx
[skip] HYPRO.OL: missing xlsx


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)


[skip] ICP1V.HE: missing xlsx
[skip] IDEX.OL: missing xlsx
[skip] ILKKA2.HE: missing xlsx
[skip] IMMUN.ST: missing xlsx
[skip] INVEST.HE: missing xlsx
[skip] IOX.OL: missing xlsx
[skip] IRLABa.ST: missing xlsx
[skip] ISOFOL.ST: missing xlsx
[skip] ISS.CO: missing xlsx
[skip] ITAB.ST: missing xlsx
[skip] ITERA.OL: missing xlsx
[skip] IWS.OL: missing xlsx
[skip] JDAN.CO: missing xlsx
[skip] JINJ.OL: missing xlsx
[skip] JM.ST: missing xlsx
[skip] JOMA.ST: missing xlsx
[skip] K2Apref.ST: missing xlsx
[skip] KABEb.ST: missing xlsx
[skip] KALMAR.HE: missing xlsx
[skip] KAMUX.HE: missing xlsx
[skip] KARNO.ST: missing xlsx
[skip] KBHL.CO: missing xlsx
[skip] KCCK.OL: missing xlsx
[skip] KCRA.HE: missing xlsx
[skip] KDV.ST: missing xlsx
[skip] KELAS.HE: missing xlsx
[skip] KEMIRA.HE: missing xlsx
[skip] KEMPOWR.HE: missing xlsx
[skip] KESKOB.HE: missing xlsx
[skip] KFASTb.ST: missing xlsx
[skip] KID.OL: missing xlsx
[skip] KIT.OL: missing xlsx
[skip] KLARAb.ST: missing xlsx
[skip] KLEEb.CO: mis

/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)


[skip] SINCH.ST: missing xlsx
[skip] SINT.ST: missing xlsx
[skip] SITOWS.HE: missing xlsx
[skip] SIVEH.ST: missing xlsx
[skip] SKAKO.CO: missing xlsx
[skip] SKAb.ST: missing xlsx
[skip] SKFb.ST: missing xlsx
[skip] SKISb.ST: missing xlsx
[skip] SLEEP.ST: missing xlsx
[skip] SLPb.ST: missing xlsx
[skip] SMCRT.OL: missing xlsx
[skip] SMOP.OL: missing xlsx
[skip] SNI.OL: missing xlsx
[skip] SOBIV.ST: missing xlsx
[skip] SOFF.OL: missing xlsx
[skip] SOFb.ST: missing xlsx
[skip] SOLARb.CO: missing xlsx
[skip] SOLTEQ.HE: missing xlsx
[skip] SOSI1.HE: missing xlsx
[skip] SPGP.CO: missing xlsx
[skip] SRV1V.HE: missing xlsx
[skip] SSABAH.HE: missing xlsx
[skip] SSABa.ST: missing xlsx
[skip] SSH1V.HE: missing xlsx
[skip] STEAV.HE: missing xlsx
[skip] STEFb.ST: missing xlsx
[skip] STEa.ST: missing xlsx
[skip] STOGR.CO: missing xlsx
[skip] STORb.ST: missing xlsx
[skip] STRAP.CO: missing xlsx
[skip] STRO.OL: missing xlsx
[skip] STWK.ST: missing xlsx
[skip] STZEb.ST: missing xlsx
[skip] SUBC.OL: mis

/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)


In [48]:
# ── Accruals extraction config ────────────────────────────────────────────────
ACC_OUT_DIR = BASE_DIR / "acc_components_extracted"

INPUT_DIR = BASE_DIR.parent / "data" / "stx_xlsx"
ACC_MAPPINGS_DIR = BASE_DIR.parent / "accruals" / "acc_mappings" / "acc_mappings_stx" 

ACC_PRIORITY_VARS = {"ACT", "CHE", "LCT", "TXP", "AT", "OANCF"}  
ACC_SUM_VARS      = {"STD", "PPEGT"}              

ACC_ALL_VARS      = ["ACT", "CHE", "LCT", "STD", "TXP", "PPEGT", "AT", "OANCF"]

def extract_firm_acc_components(xlsx_path: Path, mapping_path: Path, max_year: int = 2024):
    """Same logic as extract_firm_components but targets accruals variables."""
    mapping   = load_mapping(mapping_path)
    variables = mapping.get("variables", [])

    sheet_cache: Dict[str, pd.DataFrame] = {}
    years_union = set()
    extracted: Dict[str, pd.Series] = {}
    n_fallbacks = 0

    for var_item in variables:
        var = var_item.get("variable", "")
        if var not in ACC_ALL_VARS:
            continue

        choices = var_item.get("final_choice", [])
        if not choices:
            extracted[var] = pd.Series(dtype=float)
            continue

        label_series_list: List[pd.Series] = []
        for ch in choices:
            sheet = ch["sheet_name"]
            label = ch["row_label"]

            if sheet not in sheet_cache:
                df_raw, n = read_sheet_all_years(xlsx_path, sheet)
                n_fallbacks += n
                df_years, _ = keep_year_cols_upto(df_raw, max_year)
                sheet_cache[sheet] = df_years

            s = extract_label_series_with_duplicates(sheet_cache[sheet], label)
            if not s.empty:
                label_series_list.append(s)

        if var in ACC_SUM_VARS:
            s_var = sum_across_labels(label_series_list)
        else:
            s_var = priority_across_labels(label_series_list)

        extracted[var] = s_var
        years_union.update(list(s_var.index))

    years_sorted = sorted([y for y in years_union if isinstance(y, int)])
    df_out = pd.DataFrame(index=years_sorted)
    df_out.index.name = "Year"

    for var in ACC_ALL_VARS:
        s = extracted.get(var, pd.Series(dtype=float))
        df_out[var] = s.reindex(years_sorted)

    return df_out, n_fallbacks

In [49]:
# ── Run accruals extraction for all firms ─────────────────────────────────────
mapping_files = sorted(ACC_MAPPINGS_DIR.glob("*.json"))
print(f"Found {len(mapping_files)} mapping files")

mapping_files = sorted(ACC_MAPPINGS_DIR.glob("*.json"))
xlsx_files    = {f.stem for f in INPUT_DIR.glob("*.xlsx")}

missing = [mp.stem for mp in mapping_files if mp.stem not in xlsx_files]
print(f"{len(missing)} mappings with no matching xlsx:")
for m in missing:
    print(" ", m)

n_ok, n_skip, total_fallbacks = 0, 0, 0

for mp in mapping_files:
    firm_id   = mp.stem
    xlsx_path = INPUT_DIR / f"{firm_id}.xlsx"

    if not xlsx_path.exists():
        n_skip += 1
        continue

    df, n_fb = extract_firm_acc_components(xlsx_path, mp, max_year=MAX_YEAR)
    total_fallbacks += n_fb

    df = df.reset_index()

    company_name, industry = get_company_metadata(xlsx_path)
    sector = trbc_industry_to_sector(industry)

    df.insert(1, "Ticker",      firm_id)
    df.insert(2, "CompanyName", company_name)
    df.insert(3, "Industry",    industry)
    df.insert(4, "Sector",      sector)

    out_path = ACC_OUT_DIR / f"{firm_id}.csv"
    df.to_csv(out_path, index=False)
    n_ok += 1

print(f"Done. Extracted {n_ok} firms. Skipped {n_skip} missing workbooks.")
print(f"Header fallbacks used in {total_fallbacks} sheets.")

Found 264 mapping files
4 mappings with no matching xlsx:
  QLIRO.ST
  SAMPO.ST
  SDIPb.ST
  SFAB.ST


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin

Done. Extracted 260 firms. Skipped 4 missing workbooks.
Header fallbacks used in 0 sheets.


/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  block = hits.drop(columns=[label_col]).applymap(to_float)
/var/folders/j5/cllbm9vj70d8s736swfmxvfh0000gn/T/ipykernel_85603/2696259499.py:152: FutureWarnin